# Test-7: English to Odia Transformer

This is my from-scratch Transformer for English -> Odia translation. This notebook walks through what I built, how I trained it, and what the results look like.

Run all the cells (Shift+Enter, or Run All) to regenerate the charts yourself. I've also left a static copy of the loss curve below in case you just want to skim.

### What's in here
1. Loading the saved results
2. Model architecture and parameter count
3. The dataset and how I tokenized Odia
4. Training loss curves
5. BLEU score on the test set
6. A few sample translations
7. How translation quality drops on longer sentences
8. Attention visualization
9. Summary and the questions I expect people to ask
10. A second, bigger model I trained on a GPU, for comparison


In [ ]:
import sys
import json
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure repository root is on sys.path
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn

# Clean plot styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "sans-serif"]
plt.rcParams["axes.edgecolor"] = "#CBD5E1"
plt.rcParams["axes.linewidth"] = 0.8

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Repository root: {REPO_ROOT}")

## 1. Model Architecture

I built this as a standard encoder-decoder Transformer, layer by layer, rather than using PyTorch's `nn.Transformer`. That was the point of the assignment -- implementing attention, masking, and the encoder/decoder stack myself.

- Token embeddings + sinusoidal positional encoding
- 2 encoder blocks: self-attention, then a feed-forward layer, each with a residual connection and layer norm
- 2 decoder blocks: masked self-attention, then cross-attention over the encoder's output, then feed-forward -- same residual + norm pattern
- A final linear layer mapping to the 8,000-word Odia vocabulary

Next I check the parameter counts add up to what this architecture should give.


In [ ]:
from configs.base import (
    D_MODEL, N_HEADS, D_FF, N_ENCODER_LAYERS, N_DECODER_LAYERS,
    DROPOUT, EN_VOCAB_SIZE, OR_VOCAB_SIZE, TIE_OUTPUT_PROJECTION
)
from src.model.transformer import Seq2SeqTransformer

# Instantiate model
model = Seq2SeqTransformer(
    src_vocab_size=EN_VOCAB_SIZE,
    tgt_vocab_size=OR_VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_encoder_layers=N_ENCODER_LAYERS,
    n_decoder_layers=N_DECODER_LAYERS,
    dropout=DROPOUT,
    tie_output_projection=TIE_OUTPUT_PROJECTION
)

# Tabulate parameters by module
param_details = []
for name, p in model.named_parameters():
    param_details.append({
        "Module": name,
        "Shape": list(p.shape),
        "Parameters": p.numel(),
        "Requires Grad": p.requires_grad
    })

df_params = pd.DataFrame(param_details)

# Group high-level modules
def categorize_module(name):
    if "encoder.embeddings" in name:
        return "Source Embeddings (en)"
    elif "encoder.layers" in name:
        return "Encoder Blocks (N=2)"
    elif "decoder.embeddings" in name:
        return "Target Embeddings (or)"
    elif "decoder.layers" in name:
        return "Decoder Blocks (N=2)"
    elif "output_projection" in name:
        return "Output Projection"
    return "Other"

df_params["Category"] = df_params["Module"].apply(categorize_module)
summary_table = df_params.groupby("Category")["Parameters"].sum().reset_index()
total_params = df_params["Parameters"].sum()

print(f"=== Total Trainable Model Parameters: {total_params:,} ===")
display(summary_table.style.format({"Parameters": "{:,}"}))

# Visualization of parameter distribution
fig, ax = plt.subplots(figsize=(8, 3.8), dpi=100)
colors = ["#0D9488", "#14B8A6", "#0284C7", "#38BDF8", "#F59E0B"]
bars = ax.barh(summary_table["Category"], summary_table["Parameters"], color=colors, edgecolor="#0F172A", alpha=0.9)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 25000, bar.get_y() + bar.get_height()/2, f"{w:,} ({w/total_params:.1%})", va="center", fontsize=9, fontweight="bold")
ax.set_xlim(0, max(summary_table["Parameters"]) * 1.3)
ax.set_title("Parameter Distribution by Component", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Number of Parameters")
plt.tight_layout()
plt.show()

## 2. Dataset & Odia Tokenization

I used `ai4bharat/samanantar` for English-Odia pairs:

- Started with 58,000 raw pairs
- Cleaned the text -- Unicode normalization, stripped out junk characters, kept the marks Odia actually needs
- Dropped any pair where either side ran longer than 64 subword tokens
- Ended up with 36,000 train / 2,000 validation / 2,000 test sentences

### Odia needs a lot more tokens than English
This surprised me a bit at first: Odia's script is just more complex per character than English's, so for the same 8,000-token vocabulary, an Odia sentence needs roughly 3x as many subword tokens as the English sentence it's translating.


In [ ]:
from project_data import DATA_STATS, TOKENIZER_STATS

# 1. Dataset Split Summary
splits_data = {
    "Split": ["Train", "Validation", "Test", "Total Filtered Survivors", "Initial Candidate Pool"],
    "Sentence Pairs": [DATA_STATS["train_size"], DATA_STATS["val_size"], DATA_STATS["test_size"], DATA_STATS["retention_survivors"], DATA_STATS["candidate_pool_size"]],
    "Percentage": [
        f"{DATA_STATS['train_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['val_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['test_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['retention_rate_pct']:.1f}% retention",
        "100%"
    ]
}
display(pd.DataFrame(splits_data))

# 2. Token Length Distribution: English vs. Odia
en_stats = TOKENIZER_STATS["english"]
or_stats = TOKENIZER_STATS["odia"]

token_stats_df = pd.DataFrame({
    "Metric": ["Mean", "Median", "90th Percentile (p90)", "95th Percentile (p95)", "99th Percentile (p99)", "Max Length"],
    "English Subwords": [en_stats["mean"], en_stats["median"], en_stats["p90"], en_stats["p95"], en_stats["p99"], en_stats["max"]],
    "Odia Subwords": [or_stats["mean"], or_stats["median"], or_stats["p90"], or_stats["p95"], or_stats["p99"], or_stats["max"]]
})
display(token_stats_df)

# 3. Retention Rate vs. MAX_LEN Threshold
retention = TOKENIZER_STATS["retention_at_max_len"]
max_lens = [int(k) for k in retention.keys()]
rates = [retention[k] for k in retention.keys()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), dpi=100)

# Subplot 1: Retention Curve
ax1.plot(max_lens, rates, marker="o", color="#0D9488", linewidth=2, markersize=6)
ax1.axvline(64, color="#EF4444", linestyle="--", label=f"Chosen MAX_LEN=64 ({retention['64']}%) ")
ax1.set_title("Pair Retention Rate vs. MAX_LEN Filter", fontsize=11, fontweight="bold")
ax1.set_xlabel("Max Subword Tokens (including <SOS>/<EOS>)")
ax1.set_ylabel("Pair Retention (%)")
ax1.set_ylim(30, 105)
ax1.legend()

# Subplot 2: Subword Token Count Comparison
metrics_to_plot = ["Mean", "Median", "p90", "p95", "p99"]
x = np.arange(len(metrics_to_plot))
width = 0.35
en_vals = [en_stats["mean"], en_stats["median"], en_stats["p90"], en_stats["p95"], en_stats["p99"]]
or_vals = [or_stats["mean"], or_stats["median"], or_stats["p90"], or_stats["p95"], or_stats["p99"]]

ax2.bar(x - width/2, en_vals, width, label="English (en)", color="#3B82F6")
ax2.bar(x + width/2, or_vals, width, label="Odia (or)", color="#F97316")
ax2.set_title("Subword Token Count Comparison (8k Vocab)", fontsize=11, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_to_plot)
ax2.set_ylabel("Subwords per Sentence")
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Training

I trained for 40 epochs on a Kaggle CPU, using Adam, a Noam learning-rate schedule (warms up then decays), and label smoothing + gradient clipping + dropout to keep it from overfitting.

One thing I specifically checked for: if the decoder can accidentally "see" the word it's supposed to predict, validation loss looks unrealistically good. That's not happening here -- validation loss stays above training loss the whole way through, which is what you'd want to see.

### Loss curve
![Training & Validation Loss Curve](reports/figures/loss_curve.png)


In [ ]:
from project_data import TRAINING_HISTORY

df_history = pd.DataFrame(TRAINING_HISTORY)
df_history["train_ppl"] = np.exp(df_history["train_loss"])
df_history["val_ppl"] = np.exp(df_history["val_loss"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=100)

# Loss curves
ax1.plot(df_history["epoch"], df_history["train_loss"], label="Train Loss", color="#0D9488", linewidth=2)
ax1.plot(df_history["epoch"], df_history["val_loss"], label="Validation Loss", color="#F59E0B", linewidth=2)
ax1.set_title("Cross-Entropy Loss (with 0.1 Label Smoothing)", fontsize=11, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

# Perplexity curves
ax2.plot(df_history["epoch"], df_history["train_ppl"], label="Train Perplexity", color="#0D9488", linewidth=2)
ax2.plot(df_history["epoch"], df_history["val_ppl"], label="Validation Perplexity", color="#F59E0B", linewidth=2)
ax2.set_title("Token-Level Perplexity (PPL = exp(loss))", fontsize=11, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Perplexity")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

# Display milestones table
milestones = df_history[df_history["epoch"].isin([1, 5, 10, 20, 30, 40])].copy()
display(milestones.style.format({
    "train_loss": "{:.4f}",
    "val_loss": "{:.4f}",
    "train_ppl": "{:.2f}",
    "val_ppl": "{:.2f}"
}))

## 4. Evaluation

I evaluated on the full 2,000-sentence test set with `sacrebleu`.

**BLEU: 2.60**
- Signature: `BLEU = 2.60 22.2/4.9/1.7/0.5 (BP = 0.829, ratio = 0.842, hyp_len = 12904, ref_len = 15317)`
- An earlier run (18 epochs, no label smoothing) only got 2.19 -- training longer and adding label smoothing helped.


In [ ]:
from project_data import EVAL_RESULTS

print(f"Overall Test Corpus BLEU: {EVAL_RESULTS['bleu_score']:.2f}")
print(f"SacreBLEU Signature: {EVAL_RESULTS['bleu_signature']}")
print(f"Total Test Examples Evaluated: {EVAL_RESULTS['num_test_examples']:,}")
print(f"Evaluation Decoding Time: {EVAL_RESULTS['decode_seconds']:.1f}s")

# Precision breakdown
precisions = [22.2, 4.9, 1.7, 0.5]
ngrams = ["1-gram (Unigram)", "2-gram (Bigram)", "3-gram (Trigram)", "4-gram (4-gram)"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), dpi=100)

# Precision bars
bars = ax1.bar(ngrams, precisions, color=["#0D9488", "#14B8A6", "#06B6D4", "#0EA5E9"], edgecolor="#0F172A", alpha=0.9)
for bar in bars:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 0.6, f"{y}%", ha="center", fontweight="bold")
ax1.set_title("N-gram Precision Breakdown", fontsize=11, fontweight="bold")
ax1.set_ylabel("Precision (%)")
ax1.set_ylim(0, 27)

# Comparison: Baseline vs Enhanced
runs = ["Baseline (18 ep, no smoothing)", "Enhanced (40 ep + smoothing + rep-block)"]
bleu_vals = [2.19, 2.60]
bars2 = ax2.bar(runs, bleu_vals, color=["#94A3B8", "#0D9488"], edgecolor="#0F172A", width=0.5)
for bar in bars2:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, y + 0.08, f"BLEU {y:.2f}", ha="center", fontweight="bold")
ax2.set_title("Model Checkpoint Comparison", fontsize=11, fontweight="bold")
ax2.set_ylabel("SacreBLEU Score")
ax2.set_ylim(0, 3.2)

plt.tight_layout()
plt.show()

## 5. Sample Translations

Five examples below, including one long sentence on purpose -- I wanted to show where the model actually struggles, not just the cases where it does fine.


In [ ]:
samples = EVAL_RESULTS["samples"]
df_samples = pd.DataFrame(samples)
df_samples["Sample Type"] = ["Standard", "Standard", "Standard", "Standard", "Long Sentence (>=90th percentile)"]
df_samples = df_samples[["Sample Type", "source", "reference", "hypothesis"]]
df_samples.columns = ["Sample Type", "Source (English)", "Human Reference (Odia)", "Model Hypothesis (Odia)"]

pd.set_option("display.max_colwidth", None)
display(df_samples.style.set_properties(**{"text-align": "left"}))

## 6. Quality vs. Sentence Length

A few examples aren't really enough to judge this properly, so I grouped all 2,000 test sentences by length and looked at:
1. How BLEU changes as sentences get longer
2. How often the model gets stuck repeating itself instead of finishing


In [ ]:
from project_data import LENGTH_QUALITY_RESULTS

buckets = LENGTH_QUALITY_RESULTS["buckets_by_word_len"]
df_buckets = pd.DataFrame(buckets)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=100)

# BLEU vs sentence length
ax1.plot(df_buckets["bucket"], df_buckets["mean_bleu"], marker="s", color="#EF4444", linewidth=2, markersize=7)
for i, row in df_buckets.iterrows():
    ax1.text(i, row["mean_bleu"] + 0.35, f"{row['mean_bleu']:.1f}", ha="center", fontweight="bold", fontsize=9)
ax1.set_title("Mean Sentence BLEU vs. Source Word Count", fontsize=11, fontweight="bold")
ax1.set_xlabel("Source Sentence Word Length Bucket")
ax1.set_ylabel("Mean Sentence BLEU")
ax1.set_ylim(0, 12)
ax1.grid(True, linestyle="--", alpha=0.5)

# Repetition Rate vs sentence length
ax2.plot(df_buckets["bucket"], df_buckets["repetition_rate_pct"], marker="o", color="#8B5CF6", linewidth=2, markersize=7)
for i, row in df_buckets.iterrows():
    ax2.text(i, row["repetition_rate_pct"] + 2.5, f"{row['repetition_rate_pct']:.1f}%", ha="center", fontweight="bold", fontsize=9)
ax2.set_title("Repetition Signature Rate vs. Source Word Count", fontsize=11, fontweight="bold")
ax2.set_xlabel("Source Sentence Word Length Bucket")
ax2.set_ylabel("Repetition Rate (%)")
ax2.set_ylim(0, 90)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Overall Mean Sentence BLEU: {LENGTH_QUALITY_RESULTS['overall_mean_sentence_bleu']:.2f}")
print(f"Pearson Correlation (Word Length vs. BLEU): {LENGTH_QUALITY_RESULTS['pearson_r_wordlen_bleu']:.3f} (Significant negative correlation)")
print(f"Pearson Correlation (Subword Length vs. BLEU): {LENGTH_QUALITY_RESULTS['pearson_r_subwordlen_bleu']:.3f}")

## 7. Attention Visualization

My attention module (`src/model/attention.py`) can return its attention weights while decoding. Below is a heatmap of which English words the model was "looking at" while generating each Odia word, for one example sentence.


In [ ]:
from project_data import ATTENTION_EXAMPLES

if ATTENTION_EXAMPLES:
    example = ATTENTION_EXAMPLES[0]
    src_tokens = example["source_tokens"]
    hyp_tokens = example["hypothesis_tokens"]
    weights = np.array(example["attention_weights"])
    
    # Trim to match sequence lengths
    weights = weights[:len(hyp_tokens), :len(src_tokens)]
    
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=100)
    im = ax.imshow(weights, cmap="viridis", aspect="auto")
    
    ax.set_xticks(np.arange(len(src_tokens)))
    ax.set_yticks(np.arange(len(hyp_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(hyp_tokens, fontsize=8)
    
    ax.set_title(f"Decoder Cross-Attention Weights\n\"{example['source_text']}\"", fontsize=11, fontweight="bold", pad=12)
    ax.set_xlabel("Source English Subwords")
    ax.set_ylabel("Generated Odia Subwords")
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Attention Weight", rotation=260, labelpad=15)
    plt.tight_layout()
    plt.show()
else:
    print("No precomputed attention examples found in reports/attention_examples.json.")

## 8. Summary & Q&A

### Why isn't the translation quality higher?
It's not a bug -- I tested the attention, masking, positional encoding, and decoding separately and they all check out. It comes down to scale: this is a 4-million-parameter model trained from scratch on 36,000 sentences, on a CPU. Real translation systems like IndicTrans2 or NLLB-200 use 600M-1B+ parameters and train on tens of millions of pairs. Given what I had to work with, this BLEU score is about what I'd expect.

### What I'd do differently with more resources
Train a bigger model on a GPU (which I did -- see the next section), or start from an existing pretrained model instead of training from scratch.


---
## 9. Baseline vs. a Bigger, GPU-Trained Model

I also trained a second, larger version of the model to see how much just scaling up would help:

1. **Baseline** (this notebook): 4,005,696 params, d=128, 2+2 layers, 4 heads, trained on CPU with 36k pairs, standard Post-LN, no weight tying
2. **Scaled**: 11,469,824 params, d=256, 4+4 layers, 8 heads, trained on a Tesla T4 GPU with 60k pairs, using Pre-LN, weight tying, and a cosine LR schedule

Both checkpoints live under `checkpoints/` and get evaluated the exact same way so the comparison is fair.


In [ ]:
# Load Comparison Data and Inspect Scaled Checkpoint
comp_path = REPO_ROOT / "reports" / "model_comparison.json"
with open(comp_path, "r", encoding="utf-8") as f:
    comp_data = json.load(f)

scaled_ckpt = torch.load(REPO_ROOT / "checkpoints" / "scaled_model_best.pt", map_location="cpu")
print("=== Scaled Model Checkpoint Loaded ===")
print(f"Config: {scaled_ckpt['config']}")
print(f"Best Validation Loss: {scaled_ckpt['best_val_loss']:.4f}")
print(f"Trainable Parameters: {comp_data['summary']['scaled']['trainable_parameters']:,}")
print(f"Training Corpus: {comp_data['summary']['scaled']['training_corpus']}")
print(f"Hardware & Training Duration: {comp_data['summary']['scaled']['hardware']} ({comp_data['summary']['scaled']['training_time']})")

In [ ]:
# Display Architectural & Hyperparameter Comparison Table
df_summary = pd.DataFrame([
    {
        "Dimension": "Architecture",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["architecture"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["architecture"],
    },
    {
        "Dimension": "Hidden Dim (d_model)",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["d_model"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["d_model"],
    },
    {
        "Dimension": "Attention Heads",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["n_heads"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["n_heads"],
    },
    {
        "Dimension": "Layers (Enc + Dec)",
        "Baseline (§5.6 Course Spec)": f"{comp_data['summary']['baseline']['n_encoder_layers']} + {comp_data['summary']['baseline']['n_decoder_layers']} = 4",
        "Scaled GPU Model (Option A)": f"{comp_data['summary']['scaled']['n_encoder_layers']} + {comp_data['summary']['scaled']['n_decoder_layers']} = 8",
    },
    {
        "Dimension": "Feed-Forward (d_ff)",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["d_ff"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["d_ff"],
    },
    {
        "Dimension": "Trainable Parameters",
        "Baseline (§5.6 Course Spec)": f"{comp_data['summary']['baseline']['trainable_parameters']:,}",
        "Scaled GPU Model (Option A)": f"{comp_data['summary']['scaled']['trainable_parameters']:,}",
    },
    {
        "Dimension": "Output Weight Tying",
        "Baseline (§5.6 Course Spec)": "Disabled (Untied)",
        "Scaled GPU Model (Option A)": "Enabled (Tied Dec/Out)",
    },
    {
        "Dimension": "Training Corpus",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["training_corpus"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["training_corpus"],
    },
    {
        "Dimension": "Compute Hardware",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["hardware"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["hardware"],
    },
    {
        "Dimension": "Training Duration",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["training_time"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["training_time"],
    },
])
display(df_summary)

In [ ]:
# Display Generated Comparative Figures
from IPython.display import Image, display

print("1. Loss Curves Comparison:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_loss_curves.png")))

print("2. Parameter Breakdown Comparison:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_param_breakdown.png")))

print("3. Learning Rate Dynamics:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_lr_schedules.png")))

print("4. Comprehensive 4-Panel Comparison Dashboard:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "full_model_comparison.png")))

In [ ]:
# Display Qualitative Sample Translations Side-by-Side
df_samples = pd.DataFrame(comp_data["sample_translations_comparison"])
display(df_samples)